# 14 — Loss-function and spatial-attention ablations, station-resolved

Two comparisons, kept deliberately un-pooled: every result below is broken
out **per station**, and further by **season** and **UTC time-of-day bin**,
never collapsed into one network-wide number until explicitly noted.

**Part A — loss function.** MAE (v27, Huber) vs Probabilistic MAE
Transformer (v30-nll, Gaussian NLL), both trained at mask_ratio 0.5, both
evaluated at mr=0.5, **masked stations only** (the regime the loss functions
actually differ on — visible stations have the same residual-head shortcut
either way). Includes a "contribution of surrounding stations" section built
from the decoder's actual cross-attention weights, not a distance proxy.

**Part B — spatial-attention ablation.** MAE (v27, Huber,
mr=0.5) on its **visible** stations vs Spatially Blind
(v32-blind, Huber, mr=0.0, no spatial attention anywhere). Both Huber-only,
so this isolates the architectural ablation from the loss-function question
in Part A.

> **⚠ Data-quality caveat — read before trusting Part A's masked-station
> numbers.** `test_results/v27/best_mr0.50/predictions.pt` (2026-08-18) and
> `test_results/v30-nll/best_mr0.50/predictions.pt` (2026-08-19) both
> **predate** the 2026-08-25 evaluation-mask seeding fix documented in
> `experimental_setup.tex` §3.6 ("Known issues affecting interpretation").
> Each dump drew its own unseeded station mask, so the *specific stations*
> masked in the v27 dump are not the same ones masked in the v30-nll dump —
> **the masked-station comparison in Part A is therefore unpaired and
> provisional**, not a bug in this notebook. Regenerate both dumps with
> `run_test_cloud.sh` (same `--seed`) before quoting Part A's masked-station
> figures in the report. Part B is unaffected: v32-blind's decoder requires
> `mask_ratio=0`, so there are no masked stations to mis-pair, and the
> visible-station subset of v27's dump does not depend on which stations
> were masked that run.

**Attention section (Part A only)** needs a small `.npz` produced by
`src/scripts/extract_masked_attention.py`, which re-runs the model with
hooks that were added to `model/encoder.py` / `model/decoder.py` /
`model/mae.py` for this analysis (`forward_with_attn` /
`predict_with_decoder_attn` — diagnostic-only, never called by the trained
forward/predict path). That script needs torch + a GPU and was **not run or
tested** in the environment that wrote this notebook; run it locally first:

```
python src/scripts/extract_masked_attention.py \
    --checkpoint checkpoints/v27/best.ckpt --run_name v27 \
    --data_root /path/to/PeakWeatherDataset \
    --lead_hours 2.0 --n_windows 400 \
    --out test_results/v27/attn_mr0.50_lead2h.npz
```

If the file isn't there, the attention section prints what it expected and
skips instead of failing the notebook.

In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks", "analysis")):
    if os.path.isfile(os.path.join(_c, "common.py")):
        if _c not in sys.path: sys.path.insert(0, _c)
        break
import importlib
import common as C
importlib.reload(C)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

LEAD_MIN = 120                     # +2 h, fixed throughout this notebook
MIN_CNT  = 50                      # per-station-variable count floor for a bin to be shown

stn = C.station_table()
ns  = C.norm_stats()
VARS, STD, MEAN = ns["var_names"], ns["std"], ns["mean"]
KEEP = C.keep_mask(stn, VARS)
N = len(stn)
SEASON_LABELS = ["DJF", "MAM", "JJA", "SON"]
TOD_LABELS    = ["00\u201306 UTC", "06\u201312 UTC", "12\u201318 UTC", "18\u201324 UTC"]
print(f"{N} stations, {len(VARS)} target variables, lead=+{LEAD_MIN}min")

## Shared streaming pass

One function, reused for every (run, mask-ratio) pair in both parts: streams
`predictions.pt` once at the fixed +2h lead, accumulates **per-station**
absolute error and count for the masked / visible / all subsets, further
split by season and by 6-hour UTC time-of-day bin. Modelled directly on
`12_terrain_and_cycles.ipynb`'s `build_season_tod()`, generalised to take
`run`/`mr` as arguments and to keep the "all" subset (needed for v32-blind's
mr=0.00 dump, which has no masked stations at all).

In [ ]:
def build_station_breakdown(run, mr, lead_min=LEAD_MIN, force=False):
    """
    Per-station absolute error and count at `lead_min`, split into
    masked/visible/all subsets and further by season (4) and UTC
    time-of-day bin (4). Cached to analysis_outputs/cache/.

    Returns a dict with, for each subset s in ('msk','vis','all'):
      f"{s}_err"        (N, V)     summed |pred-obs|, pooled over all test windows
      f"{s}_cnt"        (N, V)     matching valid-sensor count
      f"{s}_err_season" (4, N, V)  same, split by season
      f"{s}_cnt_season" (4, N, V)
      f"{s}_err_tod"    (4, N, V)  same, split by UTC time-of-day bin
      f"{s}_cnt_tod"    (4, N, V)
    plus "n_windows" (scalar).
    """
    cache = os.path.join(C.CACHE, f"n14_{run}_{mr}_lead{lead_min}.npz")
    if os.path.isfile(cache) and not force:
        z = np.load(cache)
        if z["n_windows"] > 0:
            print(f"  {run}@{mr}: loaded cache ({int(z['n_windows']):,} windows)")
            return {k: z[k] for k in z.files}

    d = C.load_dump(run, mr)
    P, T, M, MI = d["preds"], d["targets"], d["masks"], d["masked_idx"]
    TH = d["target_hours"]
    grid = d["delta_steps"][0].numpy().astype(int)
    k = int(np.where(grid * 10 == lead_min)[0][0])
    Mw, V = P.shape[0], len(VARS)

    A = {}
    for s in ("msk", "vis", "all"):
        A[f"{s}_err"] = np.zeros((N, V))
        A[f"{s}_cnt"] = np.zeros((N, V))
        A[f"{s}_err_season"] = np.zeros((4, N, V))
        A[f"{s}_cnt_season"] = np.zeros((4, N, V))
        A[f"{s}_err_tod"]    = np.zeros((4, N, V))
        A[f"{s}_cnt_tod"]    = np.zeros((4, N, V))

    CH = 1000
    for a in range(0, Mw, CH):
        b = min(a + CH, Mw)
        p = P[a:b, k].numpy().astype(np.float64) * STD[None] + MEAN[None]
        o = T[a:b, k, :, :V].numpy().astype(np.float64) * STD[None] + MEAN[None]
        m = (M[a:b, k, :, :V].numpy() > 0.5) & KEEP[None]
        err = np.abs(p - o)                                        # (b-a, N, V)

        n_masked_here = MI.shape[1] if MI.ndim == 2 else 0
        if n_masked_here > 0:
            mi = MI[a:b].numpy()
            sel = np.zeros((b - a, N), bool)
            np.put_along_axis(sel, mi, True, axis=1)
        else:
            sel = np.zeros((b - a, N), bool)                       # mr=0.00: nothing masked

        th = TH[a:b, k].numpy().astype(np.float64)
        season_idx = C._season_of(th).astype(int)
        ts = pd.Timestamp("1970-01-01", tz="UTC") + pd.to_timedelta(th, unit="h")
        tod_idx = (ts.hour.values // 6).astype(int)

        subsets = [("msk", sel), ("vis", ~sel), ("all", np.ones_like(sel))]
        for tag, sub in subsets:
            w = m & sub[:, :, None]
            A[f"{tag}_err"] += (err * w).sum(axis=0)
            A[f"{tag}_cnt"] += w.sum(axis=0)
            for cax, idx in (("season", season_idx), ("tod", tod_idx)):
                np.add.at(A[f"{tag}_err_{cax}"], idx, err * w)
                np.add.at(A[f"{tag}_cnt_{cax}"], idx, w.astype(float))

    A["n_windows"] = np.array(Mw)
    np.savez_compressed(cache, **A)
    print(f"  {run}@{mr}: streamed {Mw:,} windows at lead index {k}")
    return A

def station_mae(acc, subset):
    """(N, V) per-station MAE for one subset, NaN where count < MIN_CNT."""
    err, cnt = acc[f"{subset}_err"], acc[f"{subset}_cnt"]
    return np.where(cnt > MIN_CNT, err / np.maximum(cnt, 1), np.nan)

def station_mae_binned(acc, subset, axis):
    """(4, N, V) per-station MAE split by 'season' or 'tod'."""
    err, cnt = acc[f"{subset}_err_{axis}"], acc[f"{subset}_cnt_{axis}"]
    return np.where(cnt > MIN_CNT, err / np.maximum(cnt, 1), np.nan)

def overall_norm_mae(mae_phys):
    """(N,) per-station MAE averaged across variables in normalised (per-std) units,
    so temperature (~1-2°C) and pressure (~1-2 hPa) don't dominate a raw average."""
    return np.nanmean(mae_phys / STD, axis=-1)

## Part A — Huber (v27) vs Gaussian NLL (v30-nll), mr=0.5, masked stations, +2h

*(See the caveat at the top of the notebook — these two dumps predate the
evaluation-mask seeding fix, so this comparison is unpaired/provisional.)*

In [ ]:
print("Building Part A accumulators …")
A_v27 = build_station_breakdown("v27", "mr0.50")
A_nll = build_station_breakdown("v30-nll", "mr0.50")

### A1 — Per-station table, masked stations only (no aggregation across stations)

In [ ]:
mae_v27_msk = station_mae(A_v27, "msk")     # (N, V)
mae_nll_msk = station_mae(A_nll, "msk")

tabA = pd.DataFrame({"abbr": stn.abbr.values, "terrain_class": stn.terrain_class.values,
                     "region": stn.region.values})
for i, v in enumerate(VARS):
    tabA[f"{v}_v27"]  = mae_v27_msk[:, i]
    tabA[f"{v}_nll"]  = mae_nll_msk[:, i]
tabA["overall_norm_v27"] = overall_norm_mae(mae_v27_msk)
tabA["overall_norm_nll"] = overall_norm_mae(mae_nll_msk)
tabA["gap_norm"] = tabA["overall_norm_nll"] - tabA["overall_norm_v27"]   # >0 = v27 wins
tabA["winner"] = np.where(tabA["gap_norm"] > 0, "MAE (Huber)",
                  np.where(tabA["gap_norm"] < 0, "Probabilistic MAE (NLL)", "tie"))
tabA = tabA.sort_values("gap_norm")
C.save_table(tabA, "14a_per_station_huber_vs_nll_masked")
print(f"Huber wins {int((tabA.winner=='MAE (Huber)').sum())}/{tabA.winner.notna().sum()} "
      f"stations with valid data; NLL wins "
      f"{int((tabA.winner=='Probabilistic MAE (NLL)').sum())}")
display(tabA.head(12))
display(tabA.tail(12))

### A2 — Where do the wins sit? Map + terrain/region breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
valid = tabA["gap_norm"].notna()
sc = ax.scatter(stn.loc[valid.values, "longitude"], stn.loc[valid.values, "latitude"],
                c=tabA.loc[valid, "gap_norm"], cmap="RdBu_r",
                vmin=-np.nanmax(np.abs(tabA.gap_norm)), vmax=np.nanmax(np.abs(tabA.gap_norm)),
                s=45, edgecolor="k", linewidth=0.3)
plt.colorbar(sc, ax=ax, label="gap_norm  (blue = NLL better, red = Huber better)")
ax.set_title("Masked-station skill gap, mr=0.5, +2h")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

ax = axes[1]
order = tabA.groupby("terrain_class")["gap_norm"].mean().sort_values().index
tabA.boxplot(column="gap_norm", by="terrain_class", ax=ax,
             positions=range(len(order)))
ax.axhline(0, color="grey", lw=1)
ax.set_title("Gap by terrain class"); ax.set_xlabel(""); ax.set_ylabel("gap_norm")
plt.suptitle("")
fig.tight_layout()
C.save_fig(fig, "14a_map_and_terrain_gap")
plt.show()

display(tabA.groupby("region")["gap_norm"].agg(["mean", "count"]).round(4))

### A3 — Does the winner depend on time of day?

Per-station MAE gap, split into the four 6-hour UTC bins. Heatmap: rows =
stations (sorted by mean gap), columns = time-of-day bin, colour = who wins.

In [ ]:
mae_v27_tod = station_mae_binned(A_v27, "msk", "tod")   # (4, N, V)
mae_nll_tod = station_mae_binned(A_nll, "msk", "tod")

gap_tod = np.nanmean((mae_nll_tod - mae_v27_tod) / STD[None], axis=-1)   # (4, N) normalised
order = np.argsort(np.nanmean(gap_tod, axis=0))
vmax = np.nanmax(np.abs(gap_tod))

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(gap_tod[:, order].T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(4)); ax.set_xticklabels(TOD_LABELS, rotation=30, ha="right")
ax.set_yticks(range(N)); ax.set_yticklabels(stn.abbr.values[order], fontsize=5)
ax.set_title("gap_norm by station x time-of-day (blue=NLL better, red=Huber better)")
plt.colorbar(im, ax=ax, label="gap_norm", fraction=0.03)
fig.tight_layout()
C.save_fig(fig, "14a_station_x_tod_gap")
plt.show()

tod_summary = pd.DataFrame({"tod": TOD_LABELS,
                            "mean_gap_norm": np.nanmean(gap_tod, axis=1),
                            "n_stations_favoring_huber": (gap_tod > 0).sum(axis=1),
                            "n_stations_favoring_nll": (gap_tod < 0).sum(axis=1)})
display(tod_summary)

### A4 — Does the winner depend on season (or month)?

In [ ]:
mae_v27_season = station_mae_binned(A_v27, "msk", "season")   # (4, N, V)
mae_nll_season = station_mae_binned(A_nll, "msk", "season")

gap_season = np.nanmean((mae_nll_season - mae_v27_season) / STD[None], axis=-1)  # (4, N)
vmax2 = np.nanmax(np.abs(gap_season))

fig, ax = plt.subplots(figsize=(5, 10))
im = ax.imshow(gap_season[:, order].T, cmap="RdBu_r", vmin=-vmax2, vmax=vmax2, aspect="auto")
ax.set_xticks(range(4)); ax.set_xticklabels(SEASON_LABELS)
ax.set_yticks(range(N)); ax.set_yticklabels(stn.abbr.values[order], fontsize=5)
ax.set_title("gap_norm by station x season")
plt.colorbar(im, ax=ax, label="gap_norm", fraction=0.05)
fig.tight_layout()
C.save_fig(fig, "14a_station_x_season_gap")
plt.show()

season_summary = pd.DataFrame({"season": SEASON_LABELS,
                               "mean_gap_norm": np.nanmean(gap_season, axis=1),
                               "n_stations_favoring_huber": (gap_season > 0).sum(axis=1),
                               "n_stations_favoring_nll": (gap_season < 0).sum(axis=1)})
display(season_summary)

# Monthly cross-check: coarser than season, but explicitly requested. Reuses
# the raw target_hours from the dump directly (not cached per-month, so this
# cell re-streams — acceptable at n_windows scale, skip if too slow locally).
d27 = C.load_dump("v27", "mr0.50")
grid = d27["delta_steps"][0].numpy().astype(int)
k = int(np.where(grid * 10 == LEAD_MIN)[0][0])
th = d27["target_hours"][:, k].numpy().astype(np.float64)
months = (pd.Timestamp("1970-01-01") + pd.to_timedelta(th, unit="h")).month.values
print("windows per month (v27 dump, +2h column):")
display(pd.Series(months).value_counts().sort_index())

### A5 — Contribution of surrounding stations (real decoder cross-attention)

Loads `test_results/<run>/attn_mr0.50_lead2h.npz`, produced by
`src/scripts/extract_masked_attention.py` (see the notebook header). For
every masked station, this is the actual cross-attention mass its Δ=2h
query placed on each visible station's encoder tokens — not a distance
proxy. Skips gracefully if the file hasn't been generated yet.

In [ ]:
from data.dataset import load_peakweather, build_spatial_features  # only used to
# re-derive station identity for the attention cache's `spatial` matrix, exactly
# the way common.py's station_table() verifies dump order (see its docstring).

def load_attn(run):
    path = os.path.join(C.RESULTS_ROOT, run, "attn_mr0.50_lead2h.npz")
    if not os.path.isfile(path):
        print(f"[skip] {path} not found — run extract_masked_attention.py "
              f"for {run} first (see notebook header).")
        return None
    z = np.load(path, allow_pickle=True)
    print(f"{run}: {int(z['n_windows'])} windows, "
          f"{len(z['masked_station_pos'])} masked / "
          f"{len(z['visible_station_pos'])} visible stations, "
          f"lead={float(z['lead_hours']):.2f}h")
    return {k: z[k] for k in z.files}

attn_v27 = load_attn("v27")

if attn_v27 is not None:
    # Match the cache's `spatial` matrix to station_table() row order the same
    # way common.py already verifies predictions.pt's order (see
    # station_table()'s docstring) — a mismatch here would silently mislabel
    # every station in this section, so fail loudly rather than guess.
    try:
        ds = load_peakweather(root=C.data_root())
        full_spatial, keep = build_spatial_features(ds)
        full_spatial = full_spatial.numpy()[keep.numpy() if hasattr(keep, "numpy") else keep]
        dev = np.abs(full_spatial - attn_v27["spatial"]).max()
        assert dev < 1e-3, f"attention cache station order mismatch (max dev {dev:.2e})"
        print("station order verified against attn cache")
    except Exception as e:
        print(f"[warn] could not verify station order against the attention "
              f"cache's own `spatial` field ({e}); proceeding by position only — "
              f"double-check masked_station_pos / visible_station_pos by hand "
              f"if the plot below looks wrong.")

In [ ]:
if attn_v27 is not None:
    contrib = attn_v27["contrib_mean"]              # (N_masked, N_visible)
    m_pos   = attn_v27["masked_station_pos"]
    v_pos   = attn_v27["visible_station_pos"]
    m_abbr  = stn.abbr.values[m_pos]
    v_abbr  = stn.abbr.values[v_pos]

    # Top-5 contributing visible neighbours per masked station
    top5 = []
    for i, ms in enumerate(m_abbr):
        order_i = np.argsort(-contrib[i])[:5]
        top5.append({"masked_station": ms,
                     "top_neighbours": ", ".join(
                         f"{v_abbr[j]}({contrib[i,j]:.2f})" for j in order_i)})
    top5 = pd.DataFrame(top5)
    C.save_table(top5, "14a_attn_top5_neighbours")
    display(top5)

    # Contribution vs distance: does attention mass decay with distance, as a
    # sanity check that the mechanism is doing something geographically sane?
    dists, weights = [], []
    lat = stn.latitude.values; lon = stn.longitude.values
    for i, mp in enumerate(m_pos):
        for j, vp in enumerate(v_pos):
            d_km = 111.0 * np.hypot(lat[mp] - lat[vp],
                                    (lon[mp] - lon[vp]) * np.cos(np.radians(lat[mp])))
            dists.append(d_km); weights.append(contrib[i, j])
    dists, weights = np.array(dists), np.array(weights)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(dists, weights, s=4, alpha=0.25)
    order_d = np.argsort(dists)
    bin_edges = np.linspace(0, np.percentile(dists, 95), 15)
    bin_idx = np.digitize(dists, bin_edges)
    means = [weights[bin_idx == b].mean() for b in range(1, len(bin_edges))]
    ax.plot(0.5 * (bin_edges[:-1] + bin_edges[1:]), means, "o-", color="crimson",
           label="binned mean")
    ax.set_xlabel("distance masked → visible station (km)")
    ax.set_ylabel("cross-attention weight")
    ax.set_title("v27: does decoder attention decay with distance?")
    ax.legend()
    fig.tight_layout()
    C.save_fig(fig, "14a_attn_vs_distance")
    plt.show()

In [ ]:
if attn_v27 is not None:
    # Does a more CONCENTRATED attention pattern (a few dominant neighbours)
    # correlate with lower error, vs a diffuse one? Entropy of each masked
    # station's contribution row as a concentration proxy.
    recs = pd.DataFrame(attn_v27["records"])
    p = contrib / contrib.sum(axis=1, keepdims=True).clip(min=1e-9)
    entropy = -(p * np.log(p.clip(min=1e-12))).sum(axis=1)      # (N_masked,)
    ent_by_pos = dict(zip(m_pos.tolist(), entropy.tolist()))
    recs["entropy"] = recs["masked_station_pos"].map(ent_by_pos)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(recs["entropy"], recs["abs_err"], s=6, alpha=0.2)
    ax.set_xlabel("attention entropy (low = concentrated on few neighbours)")
    ax.set_ylabel("|pred - obs| at +2h (physical units, this window)")
    ax.set_title("v27: attention concentration vs prediction error")
    fig.tight_layout()
    C.save_fig(fig, "14a_attn_entropy_vs_error")
    plt.show()
    print(f"correlation(entropy, abs_err) = "
          f"{recs[['entropy','abs_err']].corr().iloc[0,1]:.3f}  "
          f"(n={len(recs)} masked-station-windows)")

## Part B — Spatial-attention ablation, Huber only

MAE (v27, Huber, mr=0.5) restricted to its **visible** stations
vs Spatially Blind (v32-blind, Huber, mr=0.0, no spatial
attention anywhere — every station is always "visible" to itself, so its
`all` subset is the only one that exists). Both dumps postdate no seeding
concern here: v27's visible-station numbers don't depend on which stations
were masked that run, and v32-blind never masks anything by construction.

In [ ]:
print("Building Part B accumulators …")
B_v27  = build_station_breakdown("v27", "mr0.50")     # reuses Part A's cache if already built
B_blind = build_station_breakdown("v32-blind", "mr0.00")

### B1 — Per-station table (v27 visible vs v32-blind, no spatial attention)

In [ ]:
mae_v27_vis   = station_mae(B_v27, "vis")
mae_blind_all = station_mae(B_blind, "all")

tabB = pd.DataFrame({"abbr": stn.abbr.values, "terrain_class": stn.terrain_class.values,
                     "region": stn.region.values})
for i, v in enumerate(VARS):
    tabB[f"{v}_v27_vis"]   = mae_v27_vis[:, i]
    tabB[f"{v}_blind_all"] = mae_blind_all[:, i]
tabB["overall_norm_v27"]   = overall_norm_mae(mae_v27_vis)
tabB["overall_norm_blind"] = overall_norm_mae(mae_blind_all)
tabB["gap_norm"] = tabB["overall_norm_blind"] - tabB["overall_norm_v27"]  # >0 = spatial attn helps
tabB["winner"] = np.where(tabB["gap_norm"] > 0, "MAE (spatial attn)",
                  np.where(tabB["gap_norm"] < 0, "Spatially Blind", "tie"))
tabB = tabB.sort_values("gap_norm", ascending=False)
C.save_table(tabB, "14b_per_station_spatial_ablation")
print(f"Spatial attention helps at "
      f"{int((tabB.winner=='MAE (spatial attn)').sum())}/"
      f"{tabB.winner.notna().sum()} stations")
display(tabB.head(12))
display(tabB.tail(12))

### B2 — Map + terrain/region breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
validB = tabB["gap_norm"].notna()
scB = ax.scatter(stn.loc[validB.values, "longitude"], stn.loc[validB.values, "latitude"],
                 c=tabB.loc[validB, "gap_norm"], cmap="PRGn",
                 vmin=-np.nanmax(np.abs(tabB.gap_norm)), vmax=np.nanmax(np.abs(tabB.gap_norm)),
                 s=45, edgecolor="k", linewidth=0.3)
plt.colorbar(scB, ax=ax, label="gap_norm  (green = spatial attn helps)")
ax.set_title("Spatial-attention benefit, v27 visible vs v32-blind")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

ax = axes[1]
orderB = tabB.groupby("terrain_class")["gap_norm"].mean().sort_values().index
tabB.boxplot(column="gap_norm", by="terrain_class", ax=ax, positions=range(len(orderB)))
ax.axhline(0, color="grey", lw=1)
ax.set_title("Benefit by terrain class"); ax.set_xlabel(""); ax.set_ylabel("gap_norm")
plt.suptitle("")
fig.tight_layout()
C.save_fig(fig, "14b_map_and_terrain_gap")
plt.show()

display(tabB.groupby("region")["gap_norm"].agg(["mean", "count"]).round(4))

### B3 — Time-of-day and season

In [ ]:
mae_v27_vis_tod   = station_mae_binned(B_v27, "vis", "tod")
mae_blind_all_tod = station_mae_binned(B_blind, "all", "tod")
gapB_tod = np.nanmean((mae_blind_all_tod - mae_v27_vis_tod) / STD[None], axis=-1)  # (4, N)
orderB2 = np.argsort(-np.nanmean(gapB_tod, axis=0))
vmaxB = np.nanmax(np.abs(gapB_tod))

fig, axes = plt.subplots(1, 2, figsize=(11, 10))
im = axes[0].imshow(gapB_tod[:, orderB2].T, cmap="PRGn", vmin=-vmaxB, vmax=vmaxB, aspect="auto")
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(TOD_LABELS, rotation=30, ha="right")
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(stn.abbr.values[orderB2], fontsize=5)
axes[0].set_title("by time-of-day")
plt.colorbar(im, ax=axes[0], fraction=0.05)

mae_v27_vis_season   = station_mae_binned(B_v27, "vis", "season")
mae_blind_all_season = station_mae_binned(B_blind, "all", "season")
gapB_season = np.nanmean((mae_blind_all_season - mae_v27_vis_season) / STD[None], axis=-1)
im2 = axes[1].imshow(gapB_season[:, orderB2].T, cmap="PRGn", vmin=-vmaxB, vmax=vmaxB, aspect="auto")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(SEASON_LABELS)
axes[1].set_yticks([])
axes[1].set_title("by season")
plt.colorbar(im2, ax=axes[1], fraction=0.05)
fig.suptitle("Spatial-attention benefit (green=helps), station x time-of-day / season")
fig.tight_layout()
C.save_fig(fig, "14b_station_x_time_gap")
plt.show()

print("time-of-day summary:")
display(pd.DataFrame({"tod": TOD_LABELS, "mean_gap_norm": np.nanmean(gapB_tod, axis=1)}))
print("season summary:")
display(pd.DataFrame({"season": SEASON_LABELS, "mean_gap_norm": np.nanmean(gapB_season, axis=1)}))

### B4 — Contribution of surrounding stations: not applicable to v32-blind

By construction, the Spatially Blind has no cross-station
attention anywhere — its encoder drops the spatial sub-layer entirely and
its decoder is station-local (`station_local_decoder=True`), so there is no
attention weight to extract for it; the ablation's whole point is that this
pathway is closed. `extract_masked_attention.py` refuses to run against a
`station_local_decoder=True` checkpoint for exactly this reason (see its
`--abort` check). If you want the visible-station equivalent of Part A's
attention analysis for v27 alone (self-to-self and self-to-neighbour
attention among always-visible stations at mr=0.0), that would need a
separate extraction pass — not built here to keep this notebook's scope to
what was asked.